In [ ]:
#########################################################
# train_torque_model.py
#########################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import logging
import joblib

# SciPy / FFT / Stats
from scipy.signal import medfilt, butter, filtfilt
from scipy.stats import median_abs_deviation
from scipy.fft import fft, fftfreq

# Sklearn / Imblearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from sklearn.pipeline import Pipeline

# Optional SHAP (comment out if not needed)
try:
    import shap
except ImportError:
    shap = None

###############################################################################
# 1. HELPER FUNCTIONS FOR FILTERING & ENTROPY (reused by feature extraction)
###############################################################################

def butter_filter(data, ftype='high', cutoff=10, fs=100.0, order=5):
    """
    Applies a Butterworth filter (default: high-pass).
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    if normal_cutoff >= 1.0:
        # No filtering if cutoff is too high
        return data
    b, a = butter(order, normal_cutoff, btype=ftype, analog=False)
    return filtfilt(b, a, data)

def apply_filter(data, type='high', cutoff=10, fs=100.0, order=5):
    """
    Wrapper that uses the above butter_filter function.
    """
    return butter_filter(data, ftype=type, cutoff=cutoff, fs=fs, order=order)

def remove_spikes(data, threshold=7.5):
    """
    Clips data to range [-threshold, threshold].
    """
    return np.clip(data, -threshold, threshold)

def calculate_entropy(signal_data):
    """
    Basic Shannon entropy on amplitude distribution (using 32-bin histogram).
    """
    histogram, _ = np.histogram(signal_data, bins=32, density=True)
    histogram += 1e-12
    return -np.sum(histogram * np.log2(histogram))

def calculate_spectral_entropy(signal_data, fs=100.0):
    """
    Shannon entropy in frequency domain (first half of FFT).
    """
    fft_vals = np.abs(fft(signal_data))
    fft_vals = fft_vals[:len(fft_vals)//2]  # positive frequencies
    fft_sum = np.sum(fft_vals)
    if fft_sum == 0:
        return 0
    fft_vals = fft_vals / fft_sum
    fft_vals += 1e-12
    return -np.sum(fft_vals * np.log2(fft_vals))

###############################################################################
# 2. YOUR CURRENT ADVANCED FEATURE EXTRACTION FUNCTION
###############################################################################
def extract_features(torque, fs=100.0, window_size=300, threshold=7.5, kernel_size=1):
    """
    Rich time-domain & frequency-domain feature extraction.
    """
    # 1) High-pass filter
    filtered_torque = apply_filter(torque, type='high', cutoff=10, fs=fs, order=5)
    # 2) Spike removal
    torque_cleaned = remove_spikes(filtered_torque, threshold=threshold)
    # 3) Optional median filter
    torque_filtered = medfilt(torque_cleaned, kernel_size=kernel_size)
    filtered_series = pd.Series(torque_filtered)

    # Time-domain stats
    mean_val = np.mean(torque_filtered)
    median_value = np.median(torque_filtered)
    mad_val = median_abs_deviation(torque_filtered)
    std_val = np.std(torque_filtered)
    rms_val = np.sqrt(np.mean(torque_filtered**2))
    max_value = np.max(torque_filtered)
    min_value = np.min(torque_filtered)
    peak = max(abs(max_value), abs(min_value))
    mean_abs = np.mean(np.abs(torque_filtered))

    from scipy import stats
    skewness = stats.skew(torque_filtered)
    kurtosis_val = stats.kurtosis(torque_filtered, fisher=False)

    shape_factor = rms_val / mean_abs if mean_abs != 0 else 0
    crest_factor = peak / rms_val if rms_val != 0 else 0
    signal_entropy = calculate_entropy(torque_filtered)

    gradient = np.gradient(torque_filtered)
    gradient_mean = np.mean(gradient)
    gradient_std = np.std(gradient)

    rolling_median = filtered_series.rolling(window_size, min_periods=1).median()
    rolling_mad = filtered_series.rolling(window_size, min_periods=1).apply(median_abs_deviation)

    # Frequency domain
    N = len(torque_filtered)
    freqs = fftfreq(N, 1/fs)
    fft_vals = fft(torque_filtered)
    fft_magnitude = np.abs(fft_vals)

    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    fft_magnitude = fft_magnitude[pos_mask]

    if np.sum(fft_magnitude) == 0:
        # Edge case: all zeros or extremely low amplitude
        spectral_centroid = 0
        spectral_ent = 0
        peak_frequency = 0
        spectral_flatness = 0
        spectral_spread = 0
        spectral_roll_off = 0
        low_band_energy = 0
        mid_band_energy = 0
        high_band_energy = 0
        spectral_crest = 0
        spectral_flux = 0
        spectral_kurtosis = 0
        spectral_skewness = 0
        spectral_slope = 0
    else:
        spectral_centroid = np.sum(freqs * fft_magnitude) / np.sum(fft_magnitude)
        spectral_ent = calculate_spectral_entropy(torque_filtered, fs)
        peak_frequency = freqs[np.argmax(fft_magnitude)]
        spectral_flatness = (
            np.exp(np.mean(np.log(fft_magnitude))) / np.mean(fft_magnitude)
            if np.mean(fft_magnitude) != 0 else 0
        )
        spectral_spread = np.sqrt(
            np.sum(((freqs - spectral_centroid) ** 2) * fft_magnitude) / np.sum(fft_magnitude)
        )
        cumulative_spectrum = np.cumsum(fft_magnitude)
        roll_off_level = 0.85 * np.sum(fft_magnitude)
        spectral_roll_off = freqs[np.where(cumulative_spectrum >= roll_off_level)][0]
        low_band_energy = np.sum(fft_magnitude[(freqs >= 0) & (freqs < 10)])
        mid_band_energy = np.sum(fft_magnitude[(freqs >= 10) & (freqs < 30)])
        high_band_energy = np.sum(fft_magnitude[(freqs >= 30)])
        spectral_crest = (
            np.max(fft_magnitude) / np.mean(fft_magnitude)
            if np.mean(fft_magnitude) != 0 else 0
        )
        flux = np.diff(fft_magnitude)
        spectral_flux = np.sum(flux**2) if len(flux) > 0 else 0
        spectral_kurtosis = stats.kurtosis(fft_magnitude)
        spectral_skewness = stats.skew(fft_magnitude)
        spectral_slope = np.polyfit(freqs, fft_magnitude, 1)[0] if len(freqs) > 1 else 0

    features = {
        'Mean': mean_val,
        'Median': median_value,
        'MAD': mad_val,
        'Standard Deviation': std_val,
        'RMS': rms_val,
        'Shape Factor': shape_factor,
        'Crest Factor': crest_factor,
        'Entropy': signal_entropy,
        'Skewness': skewness,
        'Kurtosis': kurtosis_val,
        'Gradient Mean': gradient_mean,
        'Gradient Std Dev': gradient_std,
        'Rolling Median Mean': rolling_median.mean(),
        'Rolling MAD Mean': rolling_mad.mean(),
        'Spectral Centroid': spectral_centroid,
        'Spectral Entropy': spectral_ent,
        'Peak Frequency': peak_frequency,
        'Spectral Flatness': spectral_flatness,
        'Spectral Spread': spectral_spread,
        'Spectral Roll-off': spectral_roll_off,
        'Low Band Energy': low_band_energy,
        'Mid Band Energy': mid_band_energy,
        'High Band Energy': high_band_energy,
        'Spectral Crest': spectral_crest,
        'Spectral Flux': spectral_flux,
        'Spectral Kurtosis': spectral_kurtosis,
        'Spectral Skewness': spectral_skewness,
        'Spectral Slope': spectral_slope
    }
    return features

###############################################################################
# 3. COMPILE FEATURES FROM 'passed'/'failed' SUBFOLDERS
###############################################################################
def compile_features_from_folder(folder_path, fs=100.0):
    """
    Processes all CSV files in 'passed' and 'failed' subfolders,
    extracts features, and returns a DataFrame with features and labels.
    """
    features_list = []
    labels = []

    subfolders = {'passed': 1, 'failed': 0}
    for subfolder_name, label_val in subfolders.items():
        subfolder_path = os.path.join(folder_path, subfolder_name)
        if not os.path.isdir(subfolder_path):
            print(f"[WARNING] Subfolder '{subfolder_name}' not found in '{folder_path}'. Skipping.")
            continue

        for filename in os.listdir(subfolder_path):
            if filename.endswith('.csv') and not filename.startswith('.'):
                file_path = os.path.join(subfolder_path, filename)

                # Read CSV
                try:
                    data = pd.read_csv(file_path)
                    data.columns = data.columns.str.strip()
                    if 'N[Ncm]' not in data.columns:
                        print(f"[WARNING] No 'N[Ncm]' column in {file_path}. Skipping.")
                        continue
                except Exception as e:
                    print(f"[ERROR] Reading {file_path}: {e}")
                    continue

                # Extract torque data
                torque_data = data['N[Ncm]'].values

                # Extract features
                feats = extract_features(torque_data, fs=fs)

                # Optional filename parsing
                parts = filename.replace('.csv', '').split('_')
                datetime_str, size_str, no_str = None, None, None
                if len(parts) >= 3:
                    datetime_str = parts[0]
                    size_str = parts[1]
                    no_str = parts[2]

                feats['Datetime'] = datetime_str
                feats['Size'] = size_str
                feats['No'] = no_str

                features_list.append(feats)
                labels.append(label_val)

    df = pd.DataFrame(features_list)
    df['Label'] = labels
    return df

###############################################################################
# 4. LOAD OR BUILD FEATURES (TO AVOID RECOMPUTING)
###############################################################################
def get_or_create_features(folder_path, features_csv="compiled_features.csv", fs=100.0):
    """
    If 'compiled_features.csv' exists, load it directly.
    Otherwise, compile features from subfolders and save to CSV.
    """
    if os.path.exists(features_csv):
        print(f"[INFO] Loading features from '{features_csv}'...")
        df = pd.read_csv(features_csv)
    else:
        print("[INFO] Compiling features (this might take time)...")
        df = compile_features_from_folder(folder_path, fs=fs)
        print(f"[INFO] Saving compiled features to '{features_csv}'...")
        df.to_csv(features_csv, index=False)
    return df

###############################################################################
# 5. OPTIONAL VISUALIZATION
###############################################################################
def visualize_feature_distribution(features_df):
    """
    Plots histograms and pair plots of feature distributions.
    (Disable if dataset is large or there are many features.)
    """
    feature_cols = list(features_df.drop(columns=['Label', 'Datetime', 'Size', 'No']).columns)

    # Histograms
    features_df[feature_cols].hist(bins=30, figsize=(15, 10))
    plt.tight_layout()
    plt.show()

    # Pair Plot (can be slow for many features)
    sns.pairplot(features_df[feature_cols + ['Label']], hue='Label', diag_kind='kde')
    plt.show()

###############################################################################
# 6. TRAIN & EVALUATE MODEL
###############################################################################
def train_and_evaluate_model(folder_path,
                             features_csv="compiled_features.csv",
                             save_model_path="best_model.pkl"):
    """
    1. Either load or compile features, optionally visualize them.
    2. Split data, apply SMOTETomek.
    3. RandomizedSearchCV on multiple classifiers.
    4. Evaluate best model.
    5. Save best model if desired.

    Returns: best_model, (X_test, y_test)
    """
    # --- 6.1 Load or compile features ---
    features_df = get_or_create_features(folder_path, features_csv=features_csv, fs=100.0)

    if features_df.empty:
        raise ValueError("[ERROR] No data found. Please check your folder structure or CSV files.")

    print("Sample of extracted features:\n", features_df.head())

    # --- 6.2 (Optional) Visualize ---
    try:
        visualize_feature_distribution(features_df)
    except Exception as e:
        print(f"[WARNING] Visualization failed: {e}")

    # --- 6.3 Prepare X, y ---
    X = features_df.drop(columns=['Label', 'Datetime', 'Size', 'No'], errors='ignore')
    y = features_df['Label']

    # Remove NaNs/Infs
    X = X.replace([np.inf, -np.inf], np.nan)
    if X.isnull().values.any():
        X = X.dropna()
        y = y.loc[X.index]

    # --- 6.4 Train-Test Split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )

    # --- 6.5 Handle Class Imbalance (SMOTETomek) ---
    sm = SMOTETomek(random_state=42)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

    # --- 6.6 Define Pipeline & Parameter Distributions ---
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('classifier', LogisticRegression())
    ])

    param_dist = [
        # Logistic Regression
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
            'classifier__C': [0.01, 0.1, 1, 10],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs']
        },
        # Random Forest
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [RandomForestClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth': [None, 10, 20],
            'classifier__min_samples_split': [2, 5]
        },
        # SVC
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [SVC(probability=True, random_state=42)],
            'classifier__C': [0.1, 1, 10],
            'classifier__kernel': ['rbf', 'linear']
        },
        # K-Nearest Neighbors
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [KNeighborsClassifier()],
            'classifier__n_neighbors': [3, 5, 7],
            'classifier__weights': ['uniform', 'distance']
        },
        # Gradient Boosting
        {
            'pca__n_components': [5, 10, None],
            'classifier': [GradientBoostingClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100],
            'classifier__learning_rate': [0.01, 0.1],
            'classifier__max_depth': [3, 5]
        },
        # AdaBoost
        {
            'pca__n_components': [5, 10, None],
            'classifier': [AdaBoostClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100],
            'classifier__learning_rate': [0.01, 0.1, 1.0]
        },
        # Decision Tree (Depth 2-3)
        {
            'pca__n_components': [None],  # skip PCA typically for shallow trees
            'classifier': [DecisionTreeClassifier(random_state=42)],
            'classifier__max_depth': [2, 3],
            'classifier__min_samples_split': [2, 5],
            'classifier__criterion': ['gini', 'entropy']
        }
    ]

    # --- 6.7 Randomized Search (adjust n_iter to control run time) ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rand_search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_dist,
        cv=cv,
        scoring='accuracy',
        n_iter=20,  # reduce or increase for speed vs. thoroughness
        n_jobs=-1,
        refit=True,
        random_state=42
    )

    print("[INFO] Starting RandomizedSearchCV...")
    rand_search.fit(X_train_res, y_train_res)

    print(f"\nBest Parameters: {rand_search.best_params_}")
    print(f"Best CV Score (Accuracy): {rand_search.best_score_:.2f}")

    best_model = rand_search.best_estimator_

    # --- 6.8 Evaluate on Test Set ---
    y_pred = best_model.predict(X_test)
    if hasattr(best_model, "predict_proba"):
        y_proba = best_model.predict_proba(X_test)[:, 1]
    else:
        # fallback if no predict_proba
        y_proba = y_pred.astype(float)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=1)
    recall = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    roc_auc_val = roc_auc_score(y_test, y_proba)

    print("\nTest Set Performance:")
    print(f"Accuracy:  {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1 Score:  {f1:.2f}")
    print(f"ROC AUC:   {roc_auc_val:.2f}")

    # Classification report
    report = classification_report(y_test, y_pred, target_names=['Failed', 'Passed'], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report = df_report[['precision', 'recall', 'f1-score', 'support']]
    df_report.columns = ['Precision', 'Recall', 'F1 Score', 'Support']
    df_report = df_report.round(2)
    print("\nClassification Report:\n", df_report)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Failed', 'Passed'],
                yticklabels=['Failed', 'Passed'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC curve (AUC = {auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

    # (Optional) If best model is a Decision Tree, plot it
    if isinstance(best_model.named_steps['classifier'], DecisionTreeClassifier):
        tree_clf = best_model.named_steps['classifier']
        pca_step = best_model.named_steps['pca']
        if pca_step is None or pca_step.n_components_ is None:
            feature_names = X.columns
        else:
            # If PCA was used with some n_components
            n_pcs = pca_step.n_components_
            feature_names = [f'PC{i+1}' for i in range(n_pcs)]

        plt.figure(figsize=(16, 9))
        plot_tree(tree_clf,
                  feature_names=feature_names,
                  class_names=['Failed', 'Passed'],
                  filled=True, rounded=True)
        plt.title('Decision Tree Visualization')
        plt.show()

    # --- 6.9 Save model if desired ---
    if save_model_path:
        joblib.dump(best_model, save_model_path)
        print(f"[INFO] Best model saved to '{save_model_path}'.")

    return best_model, (X_test, y_test)

###############################################################################
# 7. MAIN EXECUTION EXAMPLE
###############################################################################
if __name__ == "__main__":
    # Adjust these paths/filenames as needed
    folder_path = r"C:\Users\chimpaleenantaphon\Documents\datafortraining_nowrap"         # where 'passed' & 'failed' subfolders are
    features_csv = r"C:\Users\chimpaleenantaphon\Documents\datafortraining_nowrap\compiled_features_nowrap.csv"               # to store/load extracted features
    model_path = "best_model_nowrap.pkl"                        # final model file

    try:
        best_model, (X_test, y_test) = train_and_evaluate_model(
            folder_path=folder_path,
            features_csv=features_csv,
            save_model_path=model_path
        )
    except Exception as e:
        logging.error(f"Training failed: {e}", exc_info=True)
    
    # (Optional) If you want to do t-SNE or SHAP, see below placeholders:

    # Example t-SNE
    """
    from sklearn.preprocessing import StandardScaler
    def visualize_tsne(X, y):
        from sklearn.manifold import TSNE
        X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X)
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y, palette='viridis')
        plt.title('t-SNE Visualization')
        plt.show()

    # Scale test data for t-SNE
    X_test_scaled = StandardScaler().fit_transform(X_test)
    visualize_tsne(X_test_scaled, y_test)
    """

    # Example SHAP
    """
    def plot_shap_values(model, X_data):
        if shap is None:
            print("SHAP not installed.")
            return
        classifier = model.named_steps['classifier']
        if isinstance(classifier, (RandomForestClassifier, GradientBoostingClassifier)):
            explainer = shap.TreeExplainer(classifier)
            shap_values = explainer.shap_values(X_data)
            shap.summary_plot(shap_values, X_data)
        else:
            print("SHAP support not implemented for this classifier.")
    
    # Use a small subset for SHAP
    X_subset = X_test.iloc[:50]
    plot_shap_values(best_model, X_subset)
    """
